<a href="https://colab.research.google.com/github/Hanu0745/ai-mentor-portfolio/blob/main/Day6_PlacementProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [8]:
if 'GEMINI_API_KEY' in os.environ:
    del os.environ['GEMINI_API_KEY']
print("GEMINI_API_KEY environment variable has been unset.")

GEMINI_API_KEY environment variable has been unset.


After running the above cell, please **re-run cell `ludNn91Jkdot`**. It should now prompt you to enter the Gemini API key again.

In [10]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [11]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    """Extract a Resume JSON from raw text. Retries once on schema fail."""
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            fix_prompt = f'Fix this JSON to match schema. Errors: {e}. Original: {resp.text}'
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={'response_mime_type': 'application/json',
                        'response_schema': Resume.model_json_schema()})
            return Resume.model_validate_json(resp.text)

In [12]:
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]
print(f'Loaded {len(resumes)} sample résumés')

results = []
errors = []
for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'  [{i+1}] {parsed.name} — {len(parsed.skills)} skills')
    except Exception as e:
        errors.append((i, e))
        print(f'  [{i+1}] FAILED: {type(e).__name__}: {str(e)[:120]}')

print(f'\n{len(results)}/5 succeeded, {len(errors)} failed')

Loaded 5 sample résumés
  [1] Ravi Kumar — 6 skills
  [2] Sneha Reddy — 6 skills
  [3] Arun Pillai — 8 skills
  [4] Priya Nair — 5 skills
  [5] Karthik Sharma — 5 skills

5/5 succeeded, 0 failed


In [13]:
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Empty input: {type(e).__name__}: {str(e)[:200]}')

# Whitespace only
try:
    bad = extract_resume('   \n\n   ')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print(f'Whitespace input: {type(e).__name__}: {str(e)[:200]}')

# Garbage non-résumé text
try:
    bad = extract_resume('the quick brown fox jumps over the lazy dog')
    print('Garbage input:', bad.model_dump_json())
except Exception as e:
    print(f'Garbage input: {type(e).__name__}: {str(e)[:200]}')

Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Unexpected success: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}
Garbage input: {"name":"","email":"","phone":null,"education":[],"skills":[],"projects":[],"experience_years":0.0}


In [14]:
class JD(BaseModel):
    company: str
    role: str
    must_have_skills: List[str]
    nice_to_have_skills: List[str] = []
    min_cgpa: Optional[float] = None
    locations: List[str] = []
    package_lpa: Optional[float] = None

In [29]:
import requests
from bs4 import BeautifulSoup
import pathlib, json

def fetch_jd(url, max_chars=6000):
    """Fetch JD URL and return clean text. Returns None on block / failure."""
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        # Remove script and style tags
        for tag in soup(['script', 'style']):
            tag.decompose()
        return soup.get_text(separator='\n', strip=True)[:max_chars]
    except Exception as e:
        print(f'  Scrape failed for {url}: {e}')
        return None

# Test on one URL
test_url = 'https://amazon.jobs/en-gb/jobs/10429167/brand-specialist-italian'
text = fetch_jd(test_url)
if text:
    print(f'Got {len(text)} chars')
    print(text[:300])
else:
    print('Scrape blocked. Will use cached set.')


Got 4517 chars
Brand Specialist - Italian - Job ID: 10429167 | Amazon.jobs
Skip to main content
×
Home
Teams
Locations
Job Categories
My career
My applications
My profile
Account security
Settings
Sign out
Resources
Disability accommodations
Benefits
Inclusive experiences
Interview tips
Leadership principles
Worki


In [19]:
def normalise_jd(text: str) -> JD:
    """Send JD text to Gemini, get structured JD JSON back."""
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f'Extract a JD JSON from this text:\n\n{text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': JD.model_json_schema(),
        },
    )
    return JD.model_validate_json(resp.text)

# Test on one JD text
if text:
    jd = normalise_jd(text)
    print(jd.model_dump_json(indent=2))

{
  "company": "Amazon",
  "role": "Brand Specialist - Italian",
  "must_have_skills": [
    "2+ years of corporate experience",
    "Italian language proficiency (minimum B2.2)"
  ],
  "nice_to_have_skills": [
    "MBA"
  ],
  "min_cgpa": null,
  "locations": [
    "Bengaluru, India"
  ],
  "package_lpa": null
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [31]:
import os

drive_data_path = '/content/drive/My Drive/Colab Notebooks/data'
if not os.path.exists(drive_data_path):
    os.makedirs(drive_data_path)
    print(f"Created directory: {drive_data_path}")
else:
    print(f"Directory already exists: {drive_data_path}")

Created directory: /content/drive/My Drive/Colab Notebooks/data


In [33]:
import json, pathlib

URLS = [
    # Paste your 5 assigned URLs here
    'https://amazon.jobs/en-gb/jobs/10429165/ctl-quick-commerce-last-mile-gsf-last-mile',
    'https://amazon.jobs/en-gb/jobs/10426444/risk-specialist-turkish-fdap-financial-risk-mitigation',
    'https://amazon.jobs/en-gb/jobs/10429164/data-center-technician',
    'https://amazon.jobs/en-gb/jobs/10388195/account-mgmt-associate-german-b2b-international-seller-growth',
    'https://amazon.jobs/en-gb/jobs/10388195/account-mgmt-associate-german-b2b-international-seller-growth',
]

# Updated CACHE path to Google Drive
CACHE = pathlib.Path('/content/drive/My Drive/Colab Notebooks/data/jds_cached.jsonl')
USE_CACHE = False   # set True if scraping is blocked

jds = []

if USE_CACHE and CACHE.exists():
    print(f'Using cached JDs from {CACHE}')
    for line in CACHE.read_text().splitlines():
        jds.append(JD.model_validate_json(line))
else:
    for url in URLS:
        text = fetch_jd(url)
        if text is None:
            continue
        try:
            jd = normalise_jd(text)
            jds.append(jd)
            print(f'  ✓ {jd.company} — {jd.role}')
        except Exception as e:
            print(f'  ✗ {url}: {e}')

    # Save to cache if not using cache
    with open(CACHE, 'w') as f:
        for jd_item in jds:
            f.write(jd_item.model_dump_json() + '\n')
    print(f"Saved {len(jds)} JDs to {CACHE}")

print(f'\nProcessed {len(jds)} JDs')

# Inspect first 3
for jd in jds[:3]:
    print(f'\n{jd.company} - {jd.role}')
    print(f'  Must: {jd.must_have_skills}')
    print(f'  Nice: {jd.nice_to_have_skills}')
    print(f'  CGPA: {jd.min_cgpa}, LPA: {jd.package_lpa}')

  ✓ Amazon — Channel Team Lead (Quick Commerce)
  Scrape failed for https://amazon.jobs/en-gb/jobs/10426444/risk-specialist-turkish-fdap-financial-risk-mitigation: ('Received response with content-encoding: zstd, but failed to decode it.', ZstdError('cannot use a decompressobj multiple times'))
  ✓ Amazon Data Services, Inc. — Data Center Technician
  Scrape failed for https://amazon.jobs/en-gb/jobs/10388195/account-mgmt-associate-german-b2b-international-seller-growth: ('Received response with content-encoding: zstd, but failed to decode it.', ZstdError('cannot use a decompressobj multiple times'))
  ✓ Amazon — Account Mgmt Associate - German, B2B, International Seller Growth
Saved 3 JDs to /content/drive/My Drive/Colab Notebooks/data/jds_cached.jsonl

Processed 3 JDs

Amazon - Channel Team Lead (Quick Commerce)
  Must: ['3-6 years in quick commerce, supply chain, or last-mile delivery operations', 'Proven experience in store operations', 'Team leadership experience', 'Metric tracking

In [34]:
OUT = pathlib.Path('data/jds.jsonl')
OUT.parent.mkdir(exist_ok=True)
with open(OUT, 'w') as f:
    for jd in jds:
        f.write(jd.model_dump_json() + '\n')
print(f'Wrote {len(jds)} JDs to {OUT}')

# Verify the file
with open(OUT) as f:
    for line in f:
        d = json.loads(line)
        print(f'  {d["company"]:20} | {d["role"]:30} | {len(d["must_have_skills"])} must-haves')

Wrote 3 JDs to data/jds.jsonl
  Amazon               | Channel Team Lead (Quick Commerce) | 8 must-haves
  Amazon Data Services, Inc. | Data Center Technician         | 31 must-haves
  Amazon               | Account Mgmt Associate - German, B2B, International Seller Growth | 3 must-haves
